# 07 — Visualisation des résultats après entraînement

Ce notebook charge les rapports JSON produits par les scripts d'entraînement et génère :
- un tableau comparatif des configurations (C1, C2, C3, C4, VFL)
- les courbes de convergence (loss et accuracy)
- une matrice de confusion sur le jeu de test (si un checkpoint est disponible)
- l'analyse par classe (meilleures / pires classes)

**Prérequis** : exécuter au préalable les scripts d'entraînement, par exemple :
```powershell
python scripts\prepare_data.py --config configs\feastnet_ultra_quick_cpu.yaml
python scripts\train_centralized.py --config configs\feastnet_ultra_quick_cpu.yaml
python scripts\train_federated.py --config configs\feastnet_ultra_quick_cpu.yaml
python scripts\train_distillation.py --config configs\feastnet_ultra_quick_cpu.yaml
python scripts\train_vfl.py --config configs\feastnet_ultra_quick_cpu.yaml
```

In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.metrics import confusion_matrix

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from feastnet_fl.config import load_config
from feastnet_fl.data.loaders import build_loaders_from_splits
from feastnet_fl.evaluation.metrics import compute_classification_metrics
from feastnet_fl.training.trainer import evaluate
from feastnet_fl.utils.experiment import build_teacher_model, get_device, load_saved_splits

CONFIG_PATH = PROJECT_ROOT / "configs" / "feastnet_ultra_quick_cpu.yaml"
config = load_config(str(CONFIG_PATH))
config["project_root"] = str(PROJECT_ROOT)
OUTPUT_ROOT = PROJECT_ROOT / config["paths"]["output_root"]
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures" / "notebook_analysis"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")
print(f"Dossier des rapports : {REPORT_DIR}")

## 1. Chargement des rapports JSON

In [ ]:
REPORT_SPECS = {
    "C1 — Centralisé": "centralized_results.json",
    "C2 — Fédéré IID": "federated_iid_results.json",
    "C3 — Fédéré non-IID": "federated_non_iid_results.json",
    "C4 — Distillation": "distillation_results.json",
    "VFL — Vertical": "vfl_results.json",
}

reports = {}
missing = []
for label, filename in REPORT_SPECS.items():
    path = REPORT_DIR / filename
    if path.exists():
        with path.open("r", encoding="utf-8") as handle:
            reports[label] = json.load(handle)
    else:
        missing.append(filename)

print(f"Rapports chargés : {list(reports.keys())}")
if missing:
    print("Rapports manquants (lancer les scripts correspondants) :")
    for name in missing:
        print(f"  - {name}")

## 2. Tableau comparatif

In [ ]:
def extract_summary(label: str, report: dict) -> dict:
    test_metrics = report.get("test_metrics", {})
    summary = report.get("summary", {})
    history = report.get("history", [])
    x_key = "epoch" if history and "epoch" in history[0] else "round"
    return {
        "configuration": label,
        "setup": report.get("setup", "n/a"),
        "test_accuracy": test_metrics.get("accuracy"),
        "test_macro_accuracy": test_metrics.get("macro_class_accuracy"),
        "test_loss": test_metrics.get("loss"),
        "num_parameters": report.get("num_parameters") or report.get("student_num_parameters"),
        "epochs_or_rounds": len(history),
        "x_axis": x_key,
        "best_val_accuracy": summary.get("best_val_accuracy"),
    }


summary_df = pd.DataFrame([extract_summary(label, report) for label, report in reports.items()])
if summary_df.empty:
    print("Aucun rapport disponible. Lancez d'abord les scripts d'entraînement.")
else:
    display_cols = [
        "configuration",
        "test_accuracy",
        "test_macro_accuracy",
        "num_parameters",
        "epochs_or_rounds",
    ]
    summary_df[display_cols].style.format(
        {
            "test_accuracy": "{:.2%}",
            "test_macro_accuracy": "{:.2%}",
            "num_parameters": "{:,.0f}",
        }
    )

In [ ]:
if not summary_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    sns.barplot(data=summary_df, x="configuration", y="test_accuracy", ax=axes[0], palette="Blues_d")
    axes[0].set_title("Précision globale (test)")
    axes[0].set_ylabel("Accuracy")
    axes[0].tick_params(axis="x", rotation=25)

    sns.barplot(
        data=summary_df,
        x="configuration",
        y="test_macro_accuracy",
        ax=axes[1],
        palette="Greens_d",
    )
    axes[1].set_title("Précision moyenne par classe (test)")
    axes[1].set_ylabel("Macro accuracy")
    axes[1].tick_params(axis="x", rotation=25)

    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "comparison_barplots.png", dpi=150)
    plt.show()

## 3. Courbes de convergence superposées

In [ ]:
def history_to_frame(label: str, report: dict) -> pd.DataFrame:
    history = report.get("history", [])
    if not history:
        return pd.DataFrame()
    frame = pd.DataFrame(history)
    x_key = "epoch" if "epoch" in frame.columns else "round"
    train_key = "train_loss" if "train_loss" in frame.columns else "client_loss"
    frame["configuration"] = label
    frame["x"] = frame[x_key]
    frame["train_metric"] = frame[train_key]
    return frame


history_frames = [history_to_frame(label, report) for label, report in reports.items()]
history_frames = [frame for frame in history_frames if not frame.empty]

if history_frames:
    history_df = pd.concat(history_frames, ignore_index=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for label in history_df["configuration"].unique():
        subset = history_df[history_df["configuration"] == label]
        axes[0].plot(subset["x"], subset["train_metric"], marker="o", label=label)
        axes[0].plot(subset["x"], subset["val_loss"], linestyle="--", alpha=0.7)
        axes[1].plot(subset["x"], subset["val_accuracy"], marker="o", label=label)

    axes[0].set_title("Loss — trait plein = train/client, pointillé = validation")
    axes[0].set_xlabel("Epoch / Round")
    axes[0].set_ylabel("Loss")
    axes[0].legend(fontsize=8)

    axes[1].set_title("Accuracy de validation")
    axes[1].set_xlabel("Epoch / Round")
    axes[1].set_ylabel("Val accuracy")
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "convergence_overlay.png", dpi=150)
    plt.show()
else:
    print("Aucun historique d'entraînement trouvé.")

## 4. Matrice de confusion (modèle centralisé C1)

In [ ]:
CHECKPOINT_PATH = OUTPUT_ROOT / "checkpoints" / "feastnet_centralized_best.pt"

if not CHECKPOINT_PATH.exists():
    print(f"Checkpoint introuvable : {CHECKPOINT_PATH}")
    print("Relancez train_centralized.py pour générer le modèle.")
else:
    splits = load_saved_splits(config)
    labels = sorted(pd.concat(list(splits.values()), ignore_index=True)["label"].unique())
    label_to_index = {label: idx for idx, label in enumerate(labels)}
    index_to_label = {idx: label for label, idx in label_to_index.items()}

    loaders = build_loaders_from_splits(
        splits={"test": splits["test"]},
        metadata_csv=str(PROJECT_ROOT / config["paths"]["metadata_csv"]),
        pointcloud_root=str(PROJECT_ROOT / config["paths"]["pointcloud_root"]),
        batch_size=config["training"]["batch_size"],
        num_workers=0,
        num_points=config["sampling"]["num_points"],
        normalize=config["sampling"]["normalize"],
        augment_train=False,
        use_rgb=config["model"]["use_rgb"],
        label_to_index=label_to_index,
    )

    device = get_device(config["training"]["device"])
    model = build_teacher_model(config).to(device)
    model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))

    all_targets, all_predictions = [], []
    model.eval()
    with torch.no_grad():
        for batch in loaders["test"]:
            batch = batch.to(device)
            preds = model(batch).argmax(dim=1)
            all_targets.append(batch.y.cpu())
            all_predictions.append(preds.cpu())

    targets = torch.cat(all_targets).numpy()
    predictions = torch.cat(all_predictions).numpy()
    cm = confusion_matrix(targets, predictions, labels=list(range(len(labels))))

    plt.figure(figsize=(14, 12))
    sns.heatmap(
        cm,
        xticklabels=labels,
        yticklabels=labels,
        cmap="Blues",
        cbar=True,
    )
    plt.title("Matrice de confusion — baseline centralisée (C1)")
    plt.xlabel("Prédiction")
    plt.ylabel("Vérité terrain")
    plt.xticks(rotation=75, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "confusion_matrix_c1.png", dpi=150)
    plt.show()

## 5. Accuracy par classe

In [ ]:
if CHECKPOINT_PATH.exists():
    class_metrics = []
    for class_index, label in index_to_label.items():
        mask = targets == class_index
        if mask.any():
            acc = (predictions[mask] == targets[mask]).mean()
            class_metrics.append({"label": label, "accuracy": acc, "support": int(mask.sum())})

    class_df = pd.DataFrame(class_metrics).sort_values("accuracy", ascending=False)
    print("Top 10 classes les mieux classées :")
    display(class_df.head(10))
    print("\nTop 10 classes les moins bien classées :")
    display(class_df.tail(10))

    plt.figure(figsize=(12, 5))
    sns.barplot(data=class_df, x="label", y="accuracy", color="steelblue")
    plt.title("Accuracy par classe — C1")
    plt.xticks(rotation=75, ha="right")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "per_class_accuracy_c1.png", dpi=150)
    plt.show()

## 6. Distillation — impact de α et τ

In [ ]:
sweep_path = REPORT_DIR / "distillation_sweep.csv"

if sweep_path.exists():
    sweep_df = pd.read_csv(sweep_path)
    pivot = sweep_df.pivot(index="alpha", columns="temperature", values="val_accuracy")

    plt.figure(figsize=(8, 5))
    sns.heatmap(pivot, annot=True, fmt=".2%", cmap="YlGnBu")
    plt.title("Sweep distillation — val accuracy (α × τ)")
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "distillation_sweep_heatmap.png", dpi=150)
    plt.show()
else:
    dist_report = reports.get("C4 — Distillation")
    if dist_report:
        print(
            "Sweep non disponible. Configuration actuelle : "
            f"alpha={dist_report.get('alpha')} | temperature={dist_report.get('temperature')}"
        )
        print("Pour un sweep complet, implémentez scripts/sweep_distillation.py.")
    else:
        print("Rapport de distillation non trouvé.")

## 7. Analyse cross-client (classes absentes par client non-IID)

In [ ]:
cross_client_path = REPORT_DIR / "cross_client_analysis.json"

if cross_client_path.exists():
    with cross_client_path.open("r", encoding="utf-8") as handle:
        cross_client = json.load(handle)
    cross_df = pd.DataFrame(cross_client["per_client"])
    display(cross_df)

    plt.figure(figsize=(8, 4))
    sns.barplot(data=cross_df, x="client", y="accuracy_on_absent_classes", palette="Oranges_d")
    plt.title("Accuracy sur classes absentes du client (HFL non-IID)")
    plt.ylim(0, 1)
    plt.tight_layout()
    plt.savefig(FIGURE_DIR / "cross_client_absent_classes.png", dpi=150)
    plt.show()
else:
    print("Rapport cross-client non trouvé.")
    print("Implémentez src/feastnet_fl/evaluation/cross_client.py pour cette analyse.")

## 8. Export du tableau comparatif

In [ ]:
if not summary_df.empty:
    export_path = REPORT_DIR / "comparison_summary.csv"
    summary_df.to_csv(export_path, index=False)
    print(f"Tableau exporté : {export_path}")
    print(f"Figures sauvegardées dans : {FIGURE_DIR}")